# 🇮🇳 IndiaData: Complete PLFS Analysis Report

**Full analysis of PLFS unit-level microdata — charts, tables, and downloadable results.**

> ✅ Validated against MoSPI official calculations (< 0.01pp difference)

### What you'll get:
- Labour indicators (LFPR, WPR, UR) by PS+SS and CWS
- Breakdowns by **Sex, Age Group, State, Sector, Education, Employment Type**
- **Publication-ready charts** — bar charts, demographic pyramids, state comparisons
- **Downloadable CSV + PNG** files

### Instructions:
1. Run each cell in order (▶ or Shift+Enter)
2. Upload your **person-level** PLFS CSV (`cperv1.csv`)
3. All analysis runs automatically

> ⚠️ Upload `cperv1.csv` (person-level), NOT `chhv1.csv` (household-level)

---
## 🔧 Step 1: Environment Setup (~3 min first time)

In [ ]:
%%capture
!pip install rpy2
%load_ext rpy2.ipython

In [ ]:
%%R
cat("Installing R packages...\n")
pkgs <- c("data.table", "survey", "srvyr", "dplyr", "yaml", "here", "fs", "ggplot2", "scales")
for (p in pkgs) {
  if (!requireNamespace(p, quietly = TRUE))
    install.packages(p, repos = "https://cloud.r-project.org/", quiet = TRUE)
}
cat("✅ All packages ready!\n")

In [ ]:
import os
REPO_DIR = "/content/IndiaData-colab"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/abhinavjnu/IndiaData-colab.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --quiet
os.chdir(REPO_DIR)
print(f"✅ Repository ready at {os.getcwd()}")

---
## 📂 Step 2: Upload PLFS Person-Level Data
Upload your `cperv1.csv` file below.

In [ ]:
from google.colab import files
import shutil

print("Upload your PLFS person-level CSV (e.g. cperv1.csv):")
uploaded = files.upload()

os.makedirs("data/raw", exist_ok=True)
for fn in uploaded.keys():
    dest = os.path.join("data/raw", fn)
    shutil.move(fn, dest)
    print(f"✅ Saved: {dest}")

---
## 📊 Step 3: Load Data, Validate & Create Survey Design

In [ ]:
%%R
suppressMessages({
  library(data.table)
  library(ggplot2)
  library(scales)
  source("R/01_config.R")
  source("R/03_survey_design.R")
  source("R/04_plfs_indicators.R")
})

# Output directory for all charts
dir.create("outputs", showWarnings = FALSE)
dir.create("outputs/charts", showWarnings = FALSE)
dir.create("outputs/tables", showWarnings = FALSE)

# ── Load data ──
csv_files <- list.files("data/raw", pattern = "\\.csv$", full.names = TRUE)
person_files <- grep("cper|person|per_", csv_files, ignore.case = TRUE, value = TRUE)
data_path <- if (length(person_files) > 0) person_files[1] else csv_files[1]

cat(sprintf("📂 Loading: %s\n", basename(data_path)))
plfs <- fread(data_path, showProgress = FALSE)
cat(sprintf("   %s records × %d columns\n\n", format(nrow(plfs), big.mark=","), ncol(plfs)))

# ── Validate ──
cols <- names(plfs)
age_var <- intersect(c("Age", "AGE", "age", "Person_Age"), cols)[1]
has_status <- any(grepl("Principal_Status|CWS_Status|Status_Code", cols))

if (is.na(age_var) || !has_status) {
  cat("❌ This is NOT a person-level file! Upload cperv1.csv instead.\n")
  cat("Columns found:", paste(cols[1:10], collapse=", "), "...\n")
  stop("Wrong file. Upload cperv1.csv (person-level).")
}

cat(sprintf("✅ Person-level confirmed (Age: %s)\n\n", age_var))

# ── Create design ──
plfs_15 <- plfs[get(age_var) >= 15]
cat(sprintf("Population 15+: %s records\n", format(nrow(plfs_15), big.mark=",")))
design <- create_plfs_design(plfs_15, level = "person")
cat("\n✅ Survey design ready!\n")

# ── Chart theme ──
theme_india <- theme_minimal(base_size = 13) +
  theme(
    plot.title = element_text(face = "bold", size = 15, hjust = 0),
    plot.subtitle = element_text(color = "grey40", size = 11),
    panel.grid.minor = element_blank(),
    panel.grid.major.y = element_line(color = "grey90"),
    legend.position = "bottom",
    plot.margin = margin(10, 15, 10, 10)
  )

india_colors <- c("#FF9933", "#138808", "#000080", "#E34234", "#6B5B95", "#9B2335")

---
## 📈 A. Overall Labour Indicators

In [ ]:
%%R -w 800 -h 400
# PS+SS (Usual Status)
lfpr_psss <- calc_lfpr(design, approach = "psss")
wpr_psss  <- calc_wpr(design, approach = "psss")
ur_psss   <- calc_unemployment_rate(design, approach = "psss")

# CWS
lfpr_cws <- calc_lfpr(design, approach = "cws")
wpr_cws  <- calc_wpr(design, approach = "cws")
ur_cws   <- calc_unemployment_rate(design, approach = "cws")

overall <- data.table(
  Indicator = rep(c("LFPR", "WPR", "UR"), 2),
  Approach = c(rep("PS+SS (Usual Status)", 3), rep("CWS (Weekly)", 3)),
  Value = c(lfpr_psss$lfpr, wpr_psss$wpr, ur_psss$ur,
            lfpr_cws$lfpr, wpr_cws$wpr, ur_cws$ur),
  SE = c(lfpr_psss$lfpr_se, wpr_psss$wpr_se, ur_psss$ur_se,
         lfpr_cws$lfpr_se, wpr_cws$wpr_se, ur_cws$ur_se)
)
overall[, Indicator := factor(Indicator, levels = c("LFPR", "WPR", "UR"))]

p1 <- ggplot(overall, aes(x = Indicator, y = Value, fill = Approach)) +
  geom_col(position = position_dodge(0.8), width = 0.7) +
  geom_errorbar(aes(ymin = Value - 1.96*SE, ymax = Value + 1.96*SE),
                position = position_dodge(0.8), width = 0.2) +
  geom_text(aes(label = sprintf("%.1f%%", Value)),
            position = position_dodge(0.8), vjust = -0.8, size = 4) +
  scale_fill_manual(values = c("#FF9933", "#138808")) +
  labs(title = "Key Labour Indicators",
       subtitle = "PLFS — Population aged 15+",
       y = "Percentage (%)", x = NULL, fill = "Approach") +
  ylim(0, max(overall$Value) * 1.15) +
  theme_india

print(p1)
ggsave("outputs/charts/01_overall_indicators.png", p1, width = 8, height = 5, dpi = 150)

fwrite(overall, "outputs/tables/01_overall_indicators.csv")
cat("\n📊 Overall indicators chart saved!\n")

---
## 👫 B. Indicators by Sex

In [ ]:
%%R -w 900 -h 450
sex_var <- detect_variable(design, "sex")

lfpr_sex <- calc_lfpr(design, by = sex_var, approach = "psss")
wpr_sex  <- calc_wpr(design, by = sex_var, approach = "psss")
ur_sex   <- calc_unemployment_rate(design, by = sex_var, approach = "psss")

sex_dt <- data.table(
  Sex = rep(c("Male", "Female"), 3),
  Indicator = c(rep("LFPR", 2), rep("WPR", 2), rep("UR", 2)),
  Value = c(lfpr_sex$lfpr[1:2], wpr_sex$wpr[1:2], ur_sex$ur[1:2])
)
sex_dt[, Indicator := factor(Indicator, levels = c("LFPR", "WPR", "UR"))]

p2 <- ggplot(sex_dt, aes(x = Indicator, y = Value, fill = Sex)) +
  geom_col(position = position_dodge(0.8), width = 0.7) +
  geom_text(aes(label = sprintf("%.1f%%", Value)),
            position = position_dodge(0.8), vjust = -0.5, size = 4) +
  scale_fill_manual(values = c("Female" = "#E34234", "Male" = "#000080")) +
  labs(title = "Labour Indicators by Sex",
       subtitle = "PS+SS (Usual Status), Age 15+",
       y = "Percentage (%)", x = NULL) +
  ylim(0, max(sex_dt$Value) * 1.15) +
  theme_india

print(p2)
ggsave("outputs/charts/02_indicators_by_sex.png", p2, width = 8, height = 5, dpi = 150)
fwrite(sex_dt, "outputs/tables/02_indicators_by_sex.csv")

---
## 🏘️ C. Rural vs Urban

In [ ]:
%%R -w 900 -h 450
sector_var <- detect_variable(design, "sector")

lfpr_sec <- calc_lfpr(design, by = sector_var, approach = "psss")
wpr_sec  <- calc_wpr(design, by = sector_var, approach = "psss")
ur_sec   <- calc_unemployment_rate(design, by = sector_var, approach = "psss")

sec_dt <- data.table(
  Sector = rep(c("Rural", "Urban"), 3),
  Indicator = c(rep("LFPR", 2), rep("WPR", 2), rep("UR", 2)),
  Value = c(lfpr_sec$lfpr[1:2], wpr_sec$wpr[1:2], ur_sec$ur[1:2])
)
sec_dt[, Indicator := factor(Indicator, levels = c("LFPR", "WPR", "UR"))]

p3 <- ggplot(sec_dt, aes(x = Indicator, y = Value, fill = Sector)) +
  geom_col(position = position_dodge(0.8), width = 0.7) +
  geom_text(aes(label = sprintf("%.1f%%", Value)),
            position = position_dodge(0.8), vjust = -0.5, size = 4) +
  scale_fill_manual(values = c("Rural" = "#138808", "Urban" = "#FF9933")) +
  labs(title = "Labour Indicators: Rural vs Urban",
       subtitle = "PS+SS (Usual Status), Age 15+",
       y = "Percentage (%)", x = NULL) +
  ylim(0, max(sec_dt$Value) * 1.15) +
  theme_india

print(p3)
ggsave("outputs/charts/03_rural_urban.png", p3, width = 8, height = 5, dpi = 150)
fwrite(sec_dt, "outputs/tables/03_rural_urban.csv")

---
## 📊 D. Age-Group Analysis

In [ ]:
%%R -w 900 -h 500
# Create age groups on the data inside the design
design_ag <- design |>
  mutate(age_group = cut(Age,
    breaks = c(14, 19, 24, 29, 34, 39, 44, 49, 54, 59, 99),
    labels = c("15-19", "20-24", "25-29", "30-34", "35-39",
               "40-44", "45-49", "50-54", "55-59", "60+")))

design_ag <- add_lf_classification(design_ag, approach = "psss")

age_result <- design_ag |>
  group_by(age_group) |>
  summarize(
    lfpr = survey_mean(is_in_lf, na.rm = TRUE) * 100,
    wpr  = survey_mean(is_employed, na.rm = TRUE) * 100,
    ur_num = survey_total(is_unemployed, na.rm = TRUE),
    lf_num = survey_total(is_in_lf, na.rm = TRUE),
    n = unweighted(n())
  ) |>
  as.data.table()
age_result[, ur := ifelse(lf_num > 0, ur_num / lf_num * 100, 0)]

# Melt for chart
age_long <- melt(age_result[, .(age_group, LFPR = lfpr, WPR = wpr, UR = ur)],
                 id.vars = "age_group", variable.name = "Indicator", value.name = "Value")

p4 <- ggplot(age_long, aes(x = age_group, y = Value, color = Indicator, group = Indicator)) +
  geom_line(linewidth = 1.2) +
  geom_point(size = 3) +
  scale_color_manual(values = c("LFPR" = "#FF9933", "WPR" = "#138808", "UR" = "#E34234")) +
  labs(title = "Labour Indicators by Age Group",
       subtitle = "PS+SS (Usual Status)",
       x = "Age Group", y = "Percentage (%)") +
  theme_india

print(p4)
ggsave("outputs/charts/04_age_group.png", p4, width = 9, height = 5, dpi = 150)
fwrite(age_result[, .(age_group, lfpr, wpr, ur, n)], "outputs/tables/04_age_group.csv")

---
## 🎓 E. Education Level Analysis

In [ ]:
%%R -w 900 -h 550
edu_labels <- c(
  "1" = "Not literate", "2" = "Literate (no school)",
  "3" = "Below primary", "4" = "Primary",
  "5" = "Middle", "6" = "Secondary",
  "7" = "Higher secondary", "8" = "Diploma/Certificate",
  "10" = "Graduate", "11" = "Post-graduate & above"
)

edu_var <- intersect(c("General_Education_Level", "Education_Level", "Education"), names(plfs))[1]
if (!is.na(edu_var)) {
  design_edu <- design |>
    mutate(edu = as.character(get(edu_var)))
  design_edu <- add_lf_classification(design_edu, approach = "psss")

  edu_result <- design_edu |>
    group_by(edu) |>
    summarize(
      lfpr = survey_mean(is_in_lf, na.rm = TRUE) * 100,
      wpr  = survey_mean(is_employed, na.rm = TRUE) * 100,
      ur_num = survey_total(is_unemployed, na.rm = TRUE),
      lf_num = survey_total(is_in_lf, na.rm = TRUE),
      n = unweighted(n())
    ) |> as.data.table()
  edu_result[, ur := ifelse(lf_num > 0, ur_num / lf_num * 100, 0)]

  # Map labels
  edu_result[, edu_label := ifelse(edu %in% names(edu_labels), edu_labels[edu], paste("Code", edu))]
  edu_result[, edu_label := factor(edu_label, levels = edu_labels[edu_labels %in% edu_result$edu_label])]

  edu_long <- melt(edu_result[!is.na(edu_label), .(edu_label, LFPR = lfpr, WPR = wpr, UR = ur)],
                   id.vars = "edu_label", variable.name = "Indicator", value.name = "Value")

  p5 <- ggplot(edu_long, aes(x = edu_label, y = Value, fill = Indicator)) +
    geom_col(position = position_dodge(0.8), width = 0.7) +
    scale_fill_manual(values = c("LFPR" = "#FF9933", "WPR" = "#138808", "UR" = "#E34234")) +
    labs(title = "Labour Indicators by Education Level",
         subtitle = "PS+SS (Usual Status), Age 15+",
         x = NULL, y = "Percentage (%)") +
    coord_flip() +
    theme_india

  print(p5)
  ggsave("outputs/charts/05_education.png", p5, width = 10, height = 6, dpi = 150)
  fwrite(edu_result[, .(edu, edu_label, lfpr, wpr, ur, n)], "outputs/tables/05_education.csv")
} else {
  cat("⚠️ No education column found in data.\n")
}

---
## 💼 F. Employment Type Distribution

In [ ]:
%%R -w 900 -h 500
# Employment type based on Principal Status Code
emp_type_labels <- c(
  "Self-employed (own account)" = "11",
  "Employer" = "12",
  "Unpaid family worker" = "21",
  "Regular salaried" = "31",
  "Casual labour (public)" = "41",
  "Casual labour (other)" = "51"
)

ps_var <- detect_variable(design, "status_ps")
design_emp <- design |>
  filter(get(ps_var) %in% c(11, 12, 21, 31, 41, 51)) |>
  mutate(emp_type = factor(
    get(ps_var),
    levels = c(11, 12, 21, 31, 41, 51),
    labels = names(emp_type_labels)
  ))

emp_dist <- design_emp |>
  group_by(emp_type) |>
  summarize(pct = survey_mean(na.rm = TRUE) * 100, n = unweighted(n())) |>
  as.data.table()

p6 <- ggplot(emp_dist, aes(x = reorder(emp_type, pct), y = pct, fill = emp_type)) +
  geom_col(width = 0.7, show.legend = FALSE) +
  geom_text(aes(label = sprintf("%.1f%%", pct)), hjust = -0.1, size = 4) +
  scale_fill_manual(values = india_colors) +
  coord_flip() +
  labs(title = "Employment Type Distribution",
       subtitle = "Among employed persons (PS), Age 15+",
       x = NULL, y = "Share of Employed (%)") +
  scale_y_continuous(expand = expansion(mult = c(0, 0.15))) +
  theme_india

print(p6)
ggsave("outputs/charts/06_employment_type.png", p6, width = 9, height = 5, dpi = 150)
fwrite(emp_dist, "outputs/tables/06_employment_type.csv")

---
## 🗺️ G. State-Level Indicators

In [ ]:
%%R -w 900 -h 900
state_var <- detect_variable(design, "state")

lfpr_st <- calc_lfpr(design, by = state_var, approach = "psss")
wpr_st  <- calc_wpr(design, by = state_var, approach = "psss")
ur_st   <- calc_unemployment_rate(design, by = state_var, approach = "psss")

state_table <- merge(
  merge(lfpr_st[, c(state_var, "lfpr"), with=FALSE],
        wpr_st[, c(state_var, "wpr"), with=FALSE], by=state_var),
  ur_st[, c(state_var, "ur"), with=FALSE], by=state_var)
setorderv(state_table, "lfpr", order = -1)

state_names <- c(
  "1"="Jammu & Kashmir", "2"="Himachal Pradesh", "3"="Punjab",
  "4"="Chandigarh", "5"="Uttarakhand", "6"="Haryana",
  "7"="Delhi", "8"="Rajasthan", "9"="Uttar Pradesh",
  "10"="Bihar", "11"="Sikkim", "12"="Arunachal Pradesh",
  "13"="Nagaland", "14"="Manipur", "15"="Mizoram",
  "16"="Tripura", "17"="Meghalaya", "18"="Assam",
  "19"="West Bengal", "20"="Jharkhand", "21"="Odisha",
  "22"="Chhattisgarh", "23"="Madhya Pradesh", "24"="Gujarat",
  "25"="Daman & Diu", "26"="D & N Haveli", "27"="Maharashtra",
  "28"="Andhra Pradesh", "29"="Karnataka", "30"="Goa",
  "31"="Lakshadweep", "32"="Kerala", "33"="Tamil Nadu",
  "34"="Puducherry", "35"="A & N Islands", "36"="Telangana",
  "37"="Ladakh"
)

state_table[, state_name := ifelse(as.character(get(state_var)) %in% names(state_names),
                                    state_names[as.character(get(state_var))],
                                    paste("State", get(state_var)))]
state_table[, state_name := factor(state_name, levels = rev(state_name))]

p7 <- ggplot(state_table, aes(x = state_name, y = lfpr)) +
  geom_col(fill = "#FF9933", width = 0.7) +
  geom_text(aes(label = sprintf("%.1f%%", lfpr)), hjust = -0.1, size = 3) +
  coord_flip() +
  labs(title = "LFPR by State/UT",
       subtitle = "PS+SS (Usual Status), Age 15+, sorted by LFPR",
       x = NULL, y = "LFPR (%)") +
  theme_india +
  theme(axis.text.y = element_text(size = 9))

print(p7)
ggsave("outputs/charts/07_lfpr_by_state.png", p7, width = 10, height = 12, dpi = 150)
fwrite(state_table, "outputs/tables/07_state_indicators.csv")

In [ ]:
%%R -w 900 -h 900
# Unemployment rate by state
state_ur <- copy(state_table)
setorderv(state_ur, "ur", order = -1)
state_ur[, state_name := factor(state_name, levels = rev(state_ur$state_name))]

p8 <- ggplot(state_ur, aes(x = state_name, y = ur)) +
  geom_col(fill = "#E34234", width = 0.7) +
  geom_text(aes(label = sprintf("%.1f%%", ur)), hjust = -0.1, size = 3) +
  coord_flip() +
  labs(title = "Unemployment Rate by State/UT",
       subtitle = "PS+SS (Usual Status), Age 15+, sorted by UR",
       x = NULL, y = "Unemployment Rate (%)") +
  theme_india +
  theme(axis.text.y = element_text(size = 9))

print(p8)
ggsave("outputs/charts/08_ur_by_state.png", p8, width = 10, height = 12, dpi = 150)

---
## 👨‍👩‍👦 H. Sex × Sector Cross-Tabulation

In [ ]:
%%R -w 900 -h 500
sex_var <- detect_variable(design, "sex")
sector_var <- detect_variable(design, "sector")

design_cross <- add_lf_classification(design, approach = "psss")
design_cross <- design_cross |>
  mutate(
    sex_label = ifelse(get(sex_var) == 1, "Male", ifelse(get(sex_var) == 2, "Female", "Other")),
    sector_label = ifelse(get(sector_var) == 1, "Rural", "Urban")
  )

cross_result <- design_cross |>
  group_by(sex_label, sector_label) |>
  summarize(
    lfpr = survey_mean(is_in_lf, na.rm = TRUE) * 100,
    wpr  = survey_mean(is_employed, na.rm = TRUE) * 100,
    ur_num = survey_total(is_unemployed, na.rm = TRUE),
    lf_num = survey_total(is_in_lf, na.rm = TRUE),
    n = unweighted(n())
  ) |> as.data.table()
cross_result[, ur := ifelse(lf_num > 0, ur_num / lf_num * 100, 0)]
cross_result <- cross_result[sex_label %in% c("Male", "Female")]

cross_long <- melt(cross_result[, .(sex_label, sector_label, LFPR = lfpr, WPR = wpr, UR = ur)],
                   id.vars = c("sex_label", "sector_label"),
                   variable.name = "Indicator", value.name = "Value")
cross_long[, Group := paste(sex_label, sector_label, sep = " - ")]

p9 <- ggplot(cross_long, aes(x = Indicator, y = Value, fill = Group)) +
  geom_col(position = position_dodge(0.8), width = 0.7) +
  geom_text(aes(label = sprintf("%.1f", Value)),
            position = position_dodge(0.8), vjust = -0.5, size = 3.5) +
  scale_fill_manual(values = c(
    "Male - Rural" = "#000080", "Male - Urban" = "#6B5B95",
    "Female - Rural" = "#E34234", "Female - Urban" = "#FF9933"
  )) +
  labs(title = "Labour Indicators: Sex × Sector",
       subtitle = "PS+SS (Usual Status), Age 15+",
       x = NULL, y = "Percentage (%)", fill = NULL) +
  ylim(0, max(cross_long$Value) * 1.15) +
  theme_india

print(p9)
ggsave("outputs/charts/09_sex_sector_cross.png", p9, width = 9, height = 5, dpi = 150)
fwrite(cross_result[, .(sex_label, sector_label, lfpr, wpr, ur, n)], "outputs/tables/09_sex_sector.csv")

---
## 📋 I. Summary Dashboard

In [ ]:
%%R
cat("\n")
cat("╔══════════════════════════════════════════════════════════╗\n")
cat("║              PLFS ANALYSIS SUMMARY                     ║\n")
cat("╠══════════════════════════════════════════════════════════╣\n")
cat(sprintf("║  Records analysed:    %-34s║\n", format(nrow(plfs_15), big.mark=",")))
cat("║                                                        ║\n")
cat("║  ── PS+SS (Usual Status) ──                            ║\n")
cat(sprintf("║  LFPR = %5.1f%%    WPR = %5.1f%%    UR = %4.1f%%         ║\n", lfpr_psss$lfpr, wpr_psss$wpr, ur_psss$ur))
cat("║                                                        ║\n")
cat("║  ── CWS (Current Weekly Status) ──                     ║\n")
cat(sprintf("║  LFPR = %5.1f%%    WPR = %5.1f%%    UR = %4.1f%%         ║\n", lfpr_cws$lfpr, wpr_cws$wpr, ur_cws$ur))
cat("║                                                        ║\n")
cat("║  ── Male vs Female (PS+SS) ──                          ║\n")
cat(sprintf("║  Male:   LFPR = %5.1f%%   Female: LFPR = %5.1f%%       ║\n", lfpr_sex$lfpr[1], lfpr_sex$lfpr[2]))
cat("║                                                        ║\n")
cat("║  ── Rural vs Urban (PS+SS) ──                          ║\n")
cat(sprintf("║  Rural:  LFPR = %5.1f%%   Urban:  LFPR = %5.1f%%       ║\n", lfpr_sec$lfpr[1], lfpr_sec$lfpr[2]))
cat("╠══════════════════════════════════════════════════════════╣\n")
cat("║  Charts: outputs/charts/    Tables: outputs/tables/    ║\n")
cat("╚══════════════════════════════════════════════════════════╝\n")

---
## 📥 Step 4: Download Everything

In [ ]:
import zipfile, glob
from google.colab import files

with zipfile.ZipFile("PLFS_Analysis_Results.zip", "w") as zf:
    for f in glob.glob("outputs/charts/*.png"):
        zf.write(f)
    for f in glob.glob("outputs/tables/*.csv"):
        zf.write(f)

print("📦 Created: PLFS_Analysis_Results.zip")
print("Contents:")
for f in sorted(glob.glob("outputs/charts/*.png") + glob.glob("outputs/tables/*.csv")):
    print(f"  {f}")

files.download("PLFS_Analysis_Results.zip")
print("\n📥 Download started!")

---
## ℹ️ About

**Source**: [IndiaData-colab](https://github.com/abhinavjnu/IndiaData-colab) | **Validation**: < 0.01pp vs MoSPI manual calculations

### Charts Generated
| # | Chart | File |
|---|-------|------|
| 1 | Overall LFPR/WPR/UR (PS+SS vs CWS) | `01_overall_indicators.png` |
| 2 | Indicators by Sex | `02_indicators_by_sex.png` |
| 3 | Rural vs Urban | `03_rural_urban.png` |
| 4 | Age Group Trends | `04_age_group.png` |
| 5 | Education Level Analysis | `05_education.png` |
| 6 | Employment Type Distribution | `06_employment_type.png` |
| 7 | LFPR by State (sorted) | `07_lfpr_by_state.png` |
| 8 | Unemployment Rate by State | `08_ur_by_state.png` |
| 9 | Sex × Sector Cross-Tab | `09_sex_sector_cross.png` |